In [ ]:
!pip install -q transformers datasets evaluate accelerate pydub imageio-ffmpeg
!pip install -q soundfile librosa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os
from datasets import Dataset, load_from_disk
import evaluate
from transformers import (
    AutoModelForAudioClassification,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer
)
import numpy as np
import imageio_ffmpeg
from pydub import AudioSegment
import torch
import torch.nn as nn
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

AudioSegment.converter = imageio_ffmpeg.get_ffmpeg_exe()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title Config
MELD_ROOT_PATH = "/content/drive/MyDrive/MELD.Raw"
MODEL_ID = "microsoft/wavlm-base-plus"
PROCESSED_DATA_PATH = "/content/drive/MyDrive/MELD_Processed_HF"
CACHE_DIR = "/content/meld_wav_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Labels
labels = ["neutral", "surprise", "fear", "sadness", "joy", "disgust", "anger"]
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

# Initialize Feature Extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)

# --- HELPER FUNCTIONS (Kept exactly as yours) ---
def convert_to_wav(path):
    wav_path = os.path.join(CACHE_DIR, os.path.basename(path).replace(".mp4", ".wav"))
    if os.path.exists(wav_path): return wav_path
    try:
        audio = AudioSegment.from_file(path)
        audio = audio.set_frame_rate(16000).set_channels(1)
        audio.export(wav_path, format="wav")
        return wav_path
    except Exception as e: return None

def load_audio_array(path):
    wav_path = convert_to_wav(path)
    if wav_path is None: return np.zeros(16000, dtype=np.float32)
    try:
        audio = AudioSegment.from_file(wav_path)
        data = np.array(audio.get_array_of_samples())
        if audio.sample_width == 2: data = data.astype(np.float32) / 32768.0
        elif audio.sample_width == 4: data = data.astype(np.float32) / 2147483648.0
        return data
    except Exception as e: return np.zeros(16000, dtype=np.float32)

def create_dataset_from_meld(csv_filename, audio_folder_name):
    csv_path = os.path.join(MELD_ROOT_PATH, csv_filename)
    audio_folder = os.path.join(MELD_ROOT_PATH, audio_folder_name)
    df = pd.read_csv(csv_path)
    def get_path(row): return os.path.join(audio_folder, f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.mp4")
    df["file_path"] = df.apply(get_path, axis=1)
    df = df[df["file_path"].apply(os.path.exists)]
    df["label"] = df["Emotion"].str.lower().map(label2id)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    return Dataset.from_pandas(df[["file_path", "label"]])

# --- DATASET CREATION ---
raw_train = create_dataset_from_meld("train/train_sent_emo.csv", "train/train_splits")
raw_eval = create_dataset_from_meld("dev_sent_emo.csv", "dev/dev_splits_complete")

def preprocess_function(examples):
    audio_arrays = [load_audio_array(path) for path in examples["file_path"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        max_length=16000 * 8,
        truncation=True
    )
    return inputs

train_save_path = os.path.join(PROCESSED_DATA_PATH, "train")
eval_save_path = os.path.join(PROCESSED_DATA_PATH, "eval")

if os.path.exists(train_save_path):
    print("Loading pre-processed TRAIN dataset from disk...")
    encoded_train = load_from_disk(train_save_path)
else:
    print("Processing TRAIN dataset...")
    encoded_train = raw_train.map(preprocess_function, batched=True, batch_size=8, remove_columns=["file_path"], num_proc=1)
    encoded_train.save_to_disk(train_save_path)

if os.path.exists(eval_save_path):
    print("Loading pre-processed EVAL dataset from disk...")
    encoded_eval = load_from_disk(eval_save_path)
else:
    print("Processing EVAL dataset...")
    encoded_eval = raw_eval.map(preprocess_function, batched=True, batch_size=8, remove_columns=["file_path"], num_proc=1)
    encoded_eval.save_to_disk(eval_save_path)

Loading pre-processed TRAIN dataset from disk...
Loading pre-processed EVAL dataset from disk...


In [ ]:
# --- CALCULATE CLASS WEIGHTS (NEW) ---
print("Calculating class weights...")
# We use the raw_train labels to calculate balance
y_train = [x['label'] for x in raw_train]
class_counts = np.bincount(y_train)
total_samples = len(y_train)
num_classes = len(labels)

# Formula: Total / (Num_Classes * Count_of_Class)
# This gives high weight to rare classes and low weight to frequent classes
class_weights_arr = total_samples / (num_classes * class_counts)
class_weights = torch.tensor(class_weights_arr, dtype=torch.float32)

print(f"Class Counts: {class_counts}")
print(f"Computed Weights: {class_weights}")

# --- CUSTOM TRAINER (NEW) ---
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Ensure weights are on the same device as the model (GPU/CPU)
        if self.class_weights.device != model.device:
            self.class_weights = self.class_weights.to(model.device)

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

Calculating class weights...
Class Counts: [3714 1003  203  475 1301  185  823]
Computed Weights: tensor([0.2963, 1.0973, 5.4215, 2.3170, 0.8459, 5.9490, 1.3373])


In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    feature_extractor: Wav2Vec2FeatureExtractor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [feature["label"] for feature in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        batch["labels"] = torch.tensor(label_features, dtype=torch.long)
        return batch

collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)
model.freeze_feature_encoder()

Some weights of WavLMForSequenceClassification were not initialized from the model checkpoint at microsoft/wavlm-base-plus and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# --- METRICS ---
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=eval_pred.label_ids)
    f1 = f1_metric.compute(predictions=predictions, references=eval_pred.label_ids, average="weighted")
    return {**acc, **f1}

# --- TRAINING ARGUMENTS (UPDATED) ---
training_args = TrainingArguments(
    output_dir="./wavlm_meld_emotion",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4, # Effective batch size = 16
    learning_rate=2e-5,            # Lowered for stability
    num_train_epochs=10,           # Increased epochs
    warmup_ratio=0.15,             # Increased warmup
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",    # Optimize for F1
    fp16=True,
    logging_steps=50,
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to="none"
)

# Initialize Custom WeightedTrainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    compute_metrics=compute_metrics,
    tokenizer=feature_extractor,
    data_collator=collator
)

/tmp/ipython-input-1912082635.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


In [ ]:
# Pass the weights to the trainer
trainer.class_weights = class_weights

print("Starting training with Class Weights...")
trainer.train()

save_path = "/content/drive/MyDrive/Final_MELD_WavLM"
trainer.save_model(save_path)
feature_extractor.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Starting training with Class Weights...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.845400,1.938471,0.433225,0.261904
2,1.848000,1.909391,0.395223,0.319389
3,1.815600,1.901077,0.423453,0.335336
4,1.743800,1.909808,0.304017,0.303217
5,1.721800,1.902435,0.370250,0.342595
6,1.650100,1.925905,0.325733,0.324105
7,1.649500,1.941323,0.345277,0.337888
8,1.549500,1.967397,0.362649,0.346807
9,1.555500,1.996607,0.321390,0.315058
10,1.512600,1.994892,0.333333,0.325126


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Model saved to /content/drive/MyDrive/Final_MELD_WavLM
